In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from Deep_Backward_BSDE_put import DeepBackward
from PolicyIteration_put import Policy_Iteration as PI


### Set d=1 for Example 1, or d=5 / 10 for Example 2.

In [ ]:

d = 5

total_time = 1.0
n_time_steps = 50
r = 0.06
sigma = 0.4
strike = 40.0
x_0 = 40.0
train_batch_size = 2**10

torch.set_default_dtype(torch.float64)
device = torch.device("cpu")

if d == 1:
    PE_iteration = 1000
    # PE_iteration = 100
    # PI_iteration = 10
    PI_iteration = 5
    hidden_dim = 21
else:
    PE_iteration = 1500
    # PE_iteration = 300
    # PI_iteration = 13
    PI_iteration = 6
    hidden_dim = 20 + d

lambda_temp = 1
K = 10.0
# epsilon_list = [0.0, 0.2, 0.4]
epsilon_list = [0.2]

print(f"d={d}, sigma={sigma}")


In [ ]:
solver_list = [
    PI(
        d=d,
        total_time=total_time,
        n_time_steps=n_time_steps,
        K=K,
        r=r,
        sigma=sigma,
        strike=strike,
        x_0=x_0,
        lambda_temp=lambda_temp,
        epsilon=epsilon,
        device=device,
        hidden_layers=2,
        hidden_dim=hidden_dim,
        lr=0.001,
    )
    for epsilon in epsilon_list
]

for solver in solver_list:
    solver.PolicyIteration(
        PI_iteration=PI_iteration,
        PE_iteration=PE_iteration,
        batch_size=train_batch_size,
    )


In [ ]:
# bsde_n_iterations = 10000
bsde_n_iterations = 5000
bsde_batch_size = 2**10

eps_ref_value = []
for eps in epsilon_list:
    print()
    print("=" * 50)
    print(f"Computing BSDE reference for ε={eps}")
    print("=" * 50)

    bsde_solver = DeepBackward(
        d=d,
        total_time=total_time,
        n_time_steps=n_time_steps,
        K=K,
        r=r,
        sigma=sigma,
        strike=strike,
        x_0=x_0,
        lambda_temp=lambda_temp,
        epsilon=eps,
        hidden_layers=2,
        hidden_dim=hidden_dim,
        lr=0.001,
    )
    _, y0_values = bsde_solver.train(
        n_iterations=bsde_n_iterations,
        batch_size=bsde_batch_size,
        print_every=2000,
        evaluate_every=500,
    )
    ref_val = y0_values[-1]
    eps_ref_value.append(ref_val)
    print(f"Reference v(0, x0) for ε={eps}: {ref_val:.6f}")

print()
print("=" * 50)
print(f"BSDE Reference values: {eps_ref_value}")
print("=" * 50)


In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")

epsilon_colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]


def get_output_filename():
    if d == 1:
        return f"put_Y0_evolution_comparison_lambda_{lambda_temp}_penalty_{K}.pdf"
    return f"put_highdim_d{d}_arithmetic_Y0_evolution_lambda_{lambda_temp}_penalty_{K}_final.pdf"


def plot_y0_evolution(iteration=PI_iteration - 1):
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    iterations = list(range(1, iteration + 2))

    for i, solver in enumerate(solver_list):
        y0_vals = solver.pi_history["y0_values"][: iteration + 1]
        ax.plot(
            iterations,
            y0_vals,
            marker="o",
            markersize=8,
            linewidth=5,
            color=epsilon_colors[i],
            alpha=0.6,
            label=f"v(0,{x_0:.0f}) (ε={epsilon_list[i]})",
        )
        ax.axhline(
            y=eps_ref_value[i],
            color=epsilon_colors[i],
            linestyle="--",
            label=f"Reference value (ε={epsilon_list[i]})",
            alpha=0.6,
            linewidth=3,
        )

        ax.annotate(
            f"{eps_ref_value[i]:.3f}",
            xy=(iterations[0], eps_ref_value[i]),
            xytext=(-20, 1),
            textcoords="offset points",
            fontsize=35,
            fontweight="bold",
            color=epsilon_colors[i],
            alpha=0.85,
            va="bottom",
            ha="left",
            annotation_clip=True,
        )

        final_val = y0_vals[-1]
        final_step = iterations[-1]
        ax.scatter([final_step], [final_val], color="red", s=75, zorder=6)
        ax.annotate(
            f"{final_val:.3f}",
            xy=(final_step, final_val),
            xytext=(-60, 9),
            textcoords="offset points",
            fontsize=35,
            fontweight="bold",
            color="red",
            va="bottom",
            ha="left",
        )

    ax.set_xlabel("PI Step", fontsize=37)
    ax.set_ylabel("v", fontsize=37)
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis="x", labelsize=35)
    ax.tick_params(axis="y", labelsize=35)
    ax.legend(loc="lower right", fontsize=27)

    if d > 1:
        ax.set_title(f"d={d}, basket put", fontsize=30)
    else:
        ax.set_title("Put", fontsize=30)

    plt.tight_layout()
    filename = get_output_filename()
    plt.savefig(filename, format="pdf", bbox_inches="tight")
    print()
    print(f"Saved comparison plot: {filename}")
    plt.show()
    plt.close()


plot_y0_evolution()
